In [20]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph , END, START
from langchain_core.messages import SystemMessage , HumanMessage
from typing import TypedDict ,Annotated,List
import operator
from dotenv import load_dotenv

In [21]:
load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b")

In [22]:
class DeepDriveState(TypedDict):
    raw_topic:str
    reserch_blueprint:str

    summary_section:str
    pros_cons_section:str
    fact_check_section:str
    case_study_section:str

    key_takeways:Annotated[List[str],operator.add]
    excution_logs:Annotated[List[str],operator.add]

    final_report:str

In [23]:
def Topic_planner_node(state:DeepDriveState)->dict:
    """ Refine the raw topic into a strucural reserch blueprint"""
    topic=state['raw_topic']
    messge=[
        SystemMessage(
            content="You are a Senior Research Director. create a brief 3-point reserch scope for this topic",
        ),
        HumanMessage(content=f"topic->  {topic}")

    ]
    response=llm.invoke(messge)
    return{
        "reserch_blueprint":response.content,
        "Execution_logs":["[planner]-> Reserch blueprint created"]
    }


In [24]:
def summarise_node(state:DeepDriveState)->dict:
    """ genrates a crisp summary and core working mechnism"""
    topic=state['raw_topic']
    blurprint=state['reserch_blueprint']
    message=[
        SystemMessage(
          content=("You are Technical Writer. write a crisp summary and explain how the topic works in simple bullts points"
                   "End your repsonse with a single line starting with Takeaway: summarizing the core essense "
          )
        ),
        HumanMessage(
            content=f"Topic:{topic}\n Blueprint:{blurprint}"
        )

    ]
    # Extract the takeway 
    takeway="Core concept Summarized"
    response=llm.invoke(message).content

    for line in response.split("\n"):
        if "Takeaway:" in line:
            takeway=line.replace("Takeaway:","").strip()
            break

    return{
        "summary_section":response,
        "Key_takeways":[f'summary Insight: {takeway}'],
        "execution_logs":["[paralle-1] Summarize node complete"]
    }

In [25]:
def pros_cons_node(state:DeepDriveState)->dict:
    """"Extracts strenghts , weeknesses, and trade-offs"""
    topic=state['pros_cons_section']
    blueprint=state['reserch_blueprint']

    message=[
        SystemMessage(
            content=("You are the System analys , provide 4 pros and cons in clean markdwon table or bullet list  "
                     "End your repsonse with a single line starting with Takeaway:about the biggest trade-off ")
        ),
        HumanMessage(
            content=(f"topic:{topic}\n RESERCH BLUEPRINT: {blueprint}")
        )
    ]
    response=llm.invoke(message).content
    takeway="trade offs evaluated"
    for line in response.split('\n'):
        if "Takeaway:" in line:
            takeway=line.replace("Takeaway:","").strip()
            break

    return{
        "pros_cons_section":response,
        "takeway":[f"Trade offs Insight: {takeway}"],
        "excution_logs":["[parallel-2] pros & cons completed"]
    }

In [26]:
# fact_check_section
def fact_check_node(state:DeepDriveState):
    topic=state['raw_topic']
    blueprint=state['reserch_blueprint']

    message=[
        SystemMessage(
            content=(
                "You are a factchecking investigator. Identify 3 common myths about this topic and bust them with hard facts. "
                "formet as Myth vs Reality"
                "End your repsonse with a single line starting with Takeaway: stating the most surprising fact. "
            )
        ),
        HumanMessage(
            content=(f"Topic:{topic}\n Reserch Blueprint {blueprint}")
        )
    ]
    response=llm.invoke(message)
    takeway="Separating Facts from Myths"
    for line in response.split("\n"):
        if "Takeaway:" in line:
            takeaway = line.replace("TAKEAWAY:", "").strip()
            break

    return {
        "fact_check_section":response,
        "key_takeways":[f"Fact check Insight: {takeway}"],
        "excution_logs":[f"[parallel-3 ] fact check section is completed"]
    }

In [27]:
def case_study_section(state:DeepDriveState)->dict:
    topic=state['raw_topic']
    blueprint=state['reserch_blueprint']
    message=[
        SystemMessage(
            content=(
                "You are an Industry Consultent . provide 2 real-world use cases and 2 critical failure risks when adopting this technoogy/topic"
                "End your repsonse with a single line starting with Takeaway: on practical adoption "
            )
        ),
        HumanMessage(
            content=(f"topic:{topic}\n Reserch blueprint{blueprint}")
        )
    ]
    response=llm.invoke(message)
    takeway="Industry cases analyzed"
    for line in response.split('\n'):
        if "Takeaway:" in line:
            takeway=line.replace("Takeaway:","").strip()

    return{
        "case_study_section":response,
        "key_takeways":[f"industry Insight:{takeway}"],
        "excution_logs":[f"[parallel-4] case study completed"]
    }

In [28]:
def report_compiler_node(state:DeepDriveState)->dict:
    topic=state['raw_topic']
    takeways_formetted="\n".join([f"{item}" for item in state["key_takeways"]])

    raw_matrial=f""""
    Topic:{topic}

    key takeways (From Parallel Reducer):
    {takeways_formetted}

    section1 -summary:
    {state['summary_section']}

    section2 -pros and cons:
    {state['pros_cons_section']}

    section3 -myths and facts
    {state['fact_check_section']}

    section4- industry case studies & Risk :
    {state['case_study_section']}

    """

    message=[
        SystemMessage(
            content="You are a Chief Editor. Combine the provided research sections into a well-structured, "
                "executive-ready Markdown report. Include an 'Executive Takeaways' box at the top, "
                "clean headings, bullet points, and a final 'Bottom Line Verdict' section."
        ),
        HumanMessage(raw_matrial)
        
    ]
    final_output=llm.invoke(message).content
    return {
        "final_report": final_output,
        "execution_logs": ["[Fan-In] Final report compiled successfully."]
    }

In [29]:
builder=StateGraph(DeepDriveState)

In [30]:
builder.add_node("Blueprint",Topic_planner_node)
builder.add_node("Sumarizer",summarise_node)
builder.add_node("check_pros_cons",pros_cons_node)
builder.add_node("fact checker",fact_check_node)
builder.add_node("Case study maker",case_study_section)
builder.add_node("report_compiler",report_compiler_node)

In [31]:
builder.add_edge(START,"Blueprint")
builder.add_edge("Blueprint","Sumarizer")
builder.add_edge("Blueprint","check_pros_cons")
builder.add_edge("Blueprint","fact checker")
builder.add_edge("Blueprint","Case study maker")

builder.add_edge("Sumarizer","report_compiler")
builder.add_edge("check_pros_cons","report_compiler")
builder.add_edge("fact checker","report_compiler")
builder.add_edge("Case study maker","report_compiler")

builder.add_edge("report_compiler",END)

In [32]:
graph=builder.compile()